[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Maurilio97-P/6d-pose-vision-workshop/blob/main/part_8_robotics_projects/24_pallet_detection_pose.ipynb)

# Notebook 24 — Pallet Detection & Fork Alignment

---

**Part 8 · Robotics Projects**

```
┌─────────────────────────────────────────────────────────────────────────┐
│   Autonomous Forklift Pallet Alignment                                  │
│                                                                         │
│   Camera ──► Detect Pallet ──► Estimate Pose ──► Fork Commands         │
│              (ArUco on        (6D: R, t)          (left/right,          │
│               pallet)                              raise/lower,         │
│                                                    advance)             │
└─────────────────────────────────────────────────────────────────────────┘
```

> **Real-world scenario**: An autonomous forklift must precisely align its forks with a pallet's entry pockets before advancing. A misalignment of even 5 cm can damage the pallet or tip the load. We'll build the complete perception and control pipeline.

In [ ]:
import sys
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    !pip install numpy matplotlib opencv-contrib-python -q
    print('Running in Google Colab — packages installed')
else:
    print('Running locally — make sure your venv is activated')

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import cv2
from dataclasses import dataclass
from typing import Optional, Tuple
from enum import Enum
print('All imports OK')

## Learning Objectives

By the end of this notebook you will:

1. Understand the **pallet geometry** and how ArUco markers map to physical fork pocket positions  
2. Compute **3D fork pocket positions** from detected ArUco pose  
3. Build a **fork alignment controller** for height, lateral, and heading correction  
4. Handle the **multi-marker averaging** strategy for robust pallet pose  
5. Implement a **safe approach sequence** (slow down near pallet)  
6. Visualize the fork entry simulation top-down and front-view

## 1 — Pallet Geometry

A standard Euro pallet (ISO 6780) is 1200mm × 800mm × 144mm.

```
TOP VIEW — Pallet with 3 ArUco Markers

  ┌─────────────────────────────────┐  ← 1200mm wide
  │                                 │
  │   [M0]          [M1]          [M2]│  ← Markers at known positions
  │   ↑                          ↑   │
  │  150mm                      150mm│
  │   from                      from │
  │   edge                      edge │
  │                                 │
  │         ←──── 1200mm ────→      │
  └─────────────────────────────────┘

FRONT VIEW — Fork Pockets

  ┌────────────────────────────────┐
  │  ████  ██████████████  ████   │  ← Pallet front face
  │  ████                  ████   │
  │  ████  ██████████████  ████   │
  ├──┘  └──┤              ├──┘  └─┤  ← Fork pockets (openings)
  │ POCKET │              │POCKET │
  │   L    │              │  R    │
  └────────┘              └───────┘
            ←─── 550mm ───→
   ←150mm→                  ←150mm→
```

### Coordinate Frame Convention

```
Pallet center marker (M1) is at origin:
  X → rightward (fork lateral direction)
  Y → upward
  Z → away from pallet face (toward forklift)

Fork pocket centers (from pallet center):
  Left pocket:  X = -275mm = -0.275m, Y = -60mm (fork height center)
  Right pocket: X = +275mm = +0.275m, Y = -60mm
```

In [ ]:
# ── Pallet & Fork Geometry ────────────────────────────────────────────────

@dataclass
class PalletModel:
    """Standard Euro pallet geometry."""
    width_m:         float = 1.200   # X dimension
    depth_m:         float = 0.800   # Y horizontal
    height_m:        float = 0.144   # Z vertical
    pocket_sep_m:    float = 0.550   # center-to-center fork pocket separation
    pocket_width_m:  float = 0.150   # width of each fork pocket opening
    pocket_height_m: float = 0.100   # height of each fork pocket opening
    pocket_depth_m:  float = 0.800   # depth (how far forks enter)

    # Fork pocket centers in pallet frame (origin = pallet center marker)
    @property
    def left_pocket_center(self):
        """Left fork pocket center in pallet frame (m)."""
        return np.array([-self.pocket_sep_m / 2, -self.pocket_height_m / 2, 0.0])

    @property
    def right_pocket_center(self):
        """Right fork pocket center in pallet frame (m)."""
        return np.array([+self.pocket_sep_m / 2, -self.pocket_height_m / 2, 0.0])

@dataclass
class ForkGeometry:
    """Forklift fork geometry."""
    fork_sep_m:     float = 0.550   # fork tip separation (center-to-center)
    fork_width_m:   float = 0.120   # each fork blade width
    fork_height_m:  float = 0.050   # each fork blade height
    fork_length_m:  float = 1.200   # fork blade length
    max_height_m:   float = 6.0     # max mast height

PALLET = PalletModel()
FORKS  = ForkGeometry()

print('Euro Pallet Model:')
print(f'  Size:            {PALLET.width_m*1000:.0f} x {PALLET.depth_m*1000:.0f} x {PALLET.height_m*1000:.0f} mm')
print(f'  Pocket sep:      {PALLET.pocket_sep_m*1000:.0f} mm (center-to-center)')
print(f'  Pocket opening:  {PALLET.pocket_width_m*1000:.0f} x {PALLET.pocket_height_m*1000:.0f} mm')
print(f'\nLeft pocket center:  {PALLET.left_pocket_center * 1000} mm')
print(f'Right pocket center: {PALLET.right_pocket_center * 1000} mm')

# ── Visualize pallet geometry ─────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Top-down view
ax = axes[0]
ax.set_facecolor('#1a1a2e')
ax.add_patch(patches.Rectangle((-PALLET.width_m/2, -PALLET.depth_m/2),
                                 PALLET.width_m, PALLET.depth_m,
                                 fc='#8B6914', ec='white', lw=1.5, label='Pallet'))
# Markers
marker_positions = [-0.45, 0.0, 0.45]   # X positions
for i, mx in enumerate(marker_positions):
    ax.add_patch(patches.Rectangle((mx-0.05, 0.1), 0.10, 0.10, fc='white', ec='black'))
    ax.text(mx, 0.155, f'M{i}', ha='center', va='center', fontsize=8, color='black')

# Fork pocket zones (front of pallet at y=+depth/2)
for pocket_x in [-PALLET.pocket_sep_m/2, PALLET.pocket_sep_m/2]:
    ax.add_patch(patches.Rectangle(
        (pocket_x - PALLET.pocket_width_m/2, -PALLET.depth_m/2 - 0.1),
        PALLET.pocket_width_m, 0.1, fc='#3498db', ec='white', lw=1, alpha=0.8))

ax.set_xlim(-0.8, 0.8); ax.set_ylim(-0.6, 0.6)
ax.set_aspect('equal')
ax.set_xlabel('X (m)'); ax.set_ylabel('Y (m)')
ax.set_title('Top-Down View: Pallet with ArUco Markers', fontweight='bold', color='white')
ax.tick_params(colors='white'); ax.xaxis.label.set_color('white'); ax.yaxis.label.set_color('white')
ax.spines[:].set_color('white')

# Front view
ax2 = axes[1]
ax2.set_facecolor('#1a1a2e')
# Pallet face
ax2.add_patch(patches.Rectangle((-PALLET.width_m/2, 0),
                                  PALLET.width_m, PALLET.height_m,
                                  fc='#8B6914', ec='white', lw=1.5))
# Fork pockets (dark holes)
for pocket_x in [-PALLET.pocket_sep_m/2, PALLET.pocket_sep_m/2]:
    ax2.add_patch(patches.Rectangle(
        (pocket_x - PALLET.pocket_width_m/2, 0.015),
        PALLET.pocket_width_m, PALLET.pocket_height_m,
        fc='#0a0a14', ec='#3498db', lw=1.5))
    ax2.text(pocket_x, 0.065, 'POCKET', ha='center', va='center',
             fontsize=7, color='#3498db')

# Fork blades
for fork_x in [-FORKS.fork_sep_m/2, FORKS.fork_sep_m/2]:
    ax2.add_patch(patches.Rectangle(
        (fork_x - FORKS.fork_width_m/2, 0.020),
        FORKS.fork_width_m, FORKS.fork_height_m,
        fc='#e74c3c', ec='white', lw=1, alpha=0.85))

ax2.set_xlim(-0.8, 0.8); ax2.set_ylim(-0.05, 0.30)
ax2.set_aspect('equal')
ax2.set_xlabel('X (m)'); ax2.set_ylabel('Height (m)')
ax2.set_title('Front View: Forks (red) Entering Pallet Pockets (blue)', fontweight='bold', color='white')
ax2.tick_params(colors='white'); ax2.xaxis.label.set_color('white'); ax2.yaxis.label.set_color('white')
ax2.spines[:].set_color('white')

plt.tight_layout()
plt.show()

## 2 — ArUco Marker Layout on Pallet

We place **3 ArUco markers** on the pallet face:

| Marker | ID | Position (pallet frame) | Role |
|--------|-----|--------------------------|------|
| M0 | 10 | (-450mm, +100mm, 0) | Left reference |
| M1 | 11 | (0mm, +100mm, 0) | Center reference (primary) |
| M2 | 12 | (+450mm, +100mm, 0) | Right reference |

Using 3 markers gives us:
- **Redundancy**: if one is occluded, we still have 2
- **Better pose averaging**: 3 independent pose estimates → more robust
- **Cross-validation**: if poses disagree, something is wrong

In [ ]:
# ── ArUco compatibility shim ──────────────────────────────────────────────

def get_aruco_dict(dict_id=cv2.aruco.DICT_4X4_50):
    try:
        return cv2.aruco.getPredefinedDictionary(dict_id)
    except AttributeError:
        return cv2.aruco.Dictionary_get(dict_id)

def get_aruco_params():
    try:
        return cv2.aruco.DetectorParameters()
    except AttributeError:
        return cv2.aruco.DetectorParameters_create()

ARUCO_DICT   = get_aruco_dict()
ARUCO_PARAMS = get_aruco_params()

# ── Marker layout in pallet frame ─────────────────────────────────────────
PALLET_MARKER_IDS = [10, 11, 12]  # ArUco IDs
PALLET_MARKER_SIZE_M = 0.10       # 10 cm marker

# Known 3D positions of marker centers in pallet frame (meters)
MARKER_POS_IN_PALLET = {
    10: np.array([-0.450, 0.100, 0.0]),   # M0 — left
    11: np.array([ 0.000, 0.100, 0.0]),   # M1 — center
    12: np.array([ 0.450, 0.100, 0.0]),   # M2 — right
}

def transform_point(p, R, t):
    """Transform 3D point p from one frame to another."""
    return R @ p + t

def poses_to_pallet_center(detected_markers, K, dist_coeffs):
    """
    Given detected ArUco markers on pallet, estimate the
    pallet center (marker M1) pose using available markers.

    detected_markers: dict {marker_id: (rvec, tvec)}
    Returns: (R_pallet_center, t_pallet_center) or None
    """
    translations = []
    rotations    = []

    for mid, (rvec, tvec) in detected_markers.items():
        if mid not in MARKER_POS_IN_PALLET:
            continue
        R_marker, _ = cv2.Rodrigues(rvec)
        t_marker = tvec.flatten()

        # Offset from this marker to pallet center (M1)
        offset = MARKER_POS_IN_PALLET[11] - MARKER_POS_IN_PALLET[mid]
        # Pallet center in camera frame = marker_pos + R_marker * offset
        t_center = t_marker + R_marker @ offset
        translations.append(t_center)
        rotations.append(R_marker)

    if not translations:
        return None, None

    # Average translations
    t_avg = np.mean(translations, axis=0)
    # Average rotations (simple mean — use Slerp for production)
    R_avg = np.mean(rotations, axis=0)
    # Re-orthogonalize via SVD
    U, _, Vt = np.linalg.svd(R_avg)
    R_avg = U @ Vt
    if np.linalg.det(R_avg) < 0:
        U[:, -1] *= -1
        R_avg = U @ Vt

    return R_avg, t_avg

# ── Test multi-marker averaging ─────────────────────────────────────────────
def euler_to_R(rx, ry, rz):
    cx, sx = np.cos(rx), np.sin(rx)
    cy, sy = np.cos(ry), np.sin(ry)
    cz, sz = np.cos(rz), np.sin(rz)
    Rx = np.array([[1,0,0],[0,cx,-sx],[0,sx,cx]])
    Ry = np.array([[cy,0,sy],[0,1,0],[-sy,0,cy]])
    Rz = np.array([[cz,-sz,0],[sz,cz,0],[0,0,1]])
    return Rz @ Ry @ Rx

# Simulate 2 visible markers (M0 and M1; M2 occluded)
R_true = euler_to_R(0.0, 0.1, 0.0)   # pallet slightly angled
t_true_center = np.array([0.05, 0.0, 1.20])  # pallet center 1.2m ahead, 5cm right

def simulate_marker_detection(marker_id):
    """Compute what the camera would see for a given pallet marker."""
    offset = MARKER_POS_IN_PALLET[marker_id] - MARKER_POS_IN_PALLET[11]  # from center
    t_marker = t_true_center + R_true @ offset
    rvec, _ = cv2.Rodrigues(R_true)
    return rvec.flatten(), t_marker

detected = {}
for mid in [10, 11]:   # M2 (ID=12) is occluded
    rvec, tvec = simulate_marker_detection(mid)
    detected[mid] = (rvec, tvec)
    print(f'Marker {mid}: tvec = {tvec.round(4)}')

K = np.array([[600, 0, 320], [0, 600, 240], [0, 0, 1]], dtype=np.float64)
dist = np.zeros(4)
R_est, t_est = poses_to_pallet_center(detected, K, dist)

print(f'\nEstimated pallet center: {t_est.round(4)}')
print(f'Ground truth center:     {t_true_center.round(4)}')
print(f'Error: {np.linalg.norm(t_est - t_true_center)*1000:.2f} mm')

## 3 — Fork Pocket Targeting

Once we know the pallet center pose `[R | t]`, we can compute exactly where the fork pockets are in 3D camera space, and therefore what corrections the forklift needs to make.

In [ ]:
# ── Fork Pocket 3D Position Calculator ────────────────────────────────────

@dataclass
class ForkAlignment:
    """Required fork corrections to enter pallet pockets."""
    lateral_error_m:  float   # positive = move right
    height_error_m:   float   # positive = raise forks
    distance_m:       float   # current distance
    heading_error_deg: float  # positive = turn right
    left_pocket_cam:  np.ndarray   # left pocket in camera frame (3,)
    right_pocket_cam: np.ndarray   # right pocket in camera frame (3,)
    is_aligned:       bool

# Alignment tolerances for fork entry
FORK_TOL_LAT_M   = 0.02   # 2 cm lateral tolerance
FORK_TOL_HEIGHT_M = 0.015  # 1.5 cm height tolerance (tighter!)
FORK_TOL_DIST_M  = 0.05   # 5 cm final approach distance tolerance
FORK_TOL_HEAD_DEG = 2.0   # 2 degree heading tolerance
FORK_DOCK_DIST_M  = 0.50  # approach to 50cm, then slowly advance

def compute_fork_alignment(R_pallet: np.ndarray, t_pallet: np.ndarray) -> ForkAlignment:
    """
    Compute fork pocket positions and required corrections.
    R_pallet, t_pallet: pallet center pose in camera frame.
    """
    # Project pocket centers to camera frame
    L = transform_point(PALLET.left_pocket_center,  R_pallet, t_pallet)
    R = transform_point(PALLET.right_pocket_center, R_pallet, t_pallet)

    # Mid-point between the two pockets (where forklift should center)
    pocket_mid = (L + R) / 2

    # Lateral: camera X of pocket midpoint
    lateral_error_m = float(pocket_mid[0])

    # Height: camera Y of pocket midpoint (negative = pocket is below camera)
    height_error_m = float(pocket_mid[1])  # positive = forks too low, need to raise

    # Distance: camera Z to pallet face
    distance_m = float(t_pallet[2])

    # Heading: extract Y rotation from pallet R
    heading_error_deg = float(np.degrees(np.arctan2(R_pallet[0, 2], R_pallet[2, 2])))

    is_aligned = (
        abs(lateral_error_m)   < FORK_TOL_LAT_M    and
        abs(height_error_m)    < FORK_TOL_HEIGHT_M  and
        abs(distance_m - FORK_DOCK_DIST_M) < FORK_TOL_DIST_M and
        abs(heading_error_deg) < FORK_TOL_HEAD_DEG
    )

    return ForkAlignment(
        lateral_error_m=lateral_error_m,
        height_error_m=height_error_m,
        distance_m=distance_m,
        heading_error_deg=heading_error_deg,
        left_pocket_cam=L,
        right_pocket_cam=R,
        is_aligned=is_aligned
    )

@dataclass
class ForkliftCommand:
    forward:  float    # m/s
    lateral:  float    # m/s (left/right drive)
    turn:     float    # rad/s
    lift:     float    # m/s (+ = raise, - = lower)
    message:  str

def generate_forklift_command(fa: ForkAlignment) -> ForkliftCommand:
    """Proportional controller for forklift fork alignment."""
    if fa.is_aligned:
        return ForkliftCommand(0.0, 0.0, 0.0, 0.0, 'FORKS ALIGNED — ADVANCE SLOWLY')

    # P-controller gains
    KP_FWD  = 0.3   # m/s per m of distance error
    KP_LAT  = 1.0
    KP_TURN = 0.05  # rad/s per degree
    KP_LIFT = 0.8   # m/s per m of height error

    MAX_FWD  = 0.15  # slow near pallet
    MAX_LAT  = 0.20
    MAX_TURN = 0.30
    MAX_LIFT = 0.10  # m/s

    dist_err = fa.distance_m - FORK_DOCK_DIST_M
    forward  = np.clip(KP_FWD  * dist_err,         -MAX_FWD,  MAX_FWD)
    lateral  = np.clip(KP_LAT  * fa.lateral_error_m, -MAX_LAT, MAX_LAT)
    turn     = np.clip(KP_TURN * fa.heading_error_deg, -MAX_TURN, MAX_TURN)
    lift     = np.clip(KP_LIFT * (-fa.height_error_m), -MAX_LIFT, MAX_LIFT)

    msg = (f'lat={fa.lateral_error_m*100:+.1f}cm  '
           f'ht={fa.height_error_m*100:+.1f}cm  '
           f'dist={fa.distance_m:.2f}m  '
           f'hdg={fa.heading_error_deg:+.1f}°')

    return ForkliftCommand(forward=forward, lateral=lateral,
                           turn=turn, lift=lift, message=msg)

# ── Test alignment scenarios ───────────────────────────────────────────────
scenarios = [
    (euler_to_R(0.0, 0.12, 0.0), np.array([0.10, 0.05, 1.50]), 'Far approach'),
    (euler_to_R(0.0, 0.05, 0.0), np.array([0.03, 0.02, 0.80]), 'Aligning'),
    (euler_to_R(0.0, 0.02, 0.0), np.array([0.01, 0.01, 0.50]), 'Nearly aligned'),
    (euler_to_R(0.0, 0.01, 0.0), np.array([0.005, 0.008, 0.50]), 'ALIGNED!'),
]

print('═' * 75)
print('  Fork Alignment Controller')
print('═' * 75)
for R_p, t_p, label in scenarios:
    fa = compute_fork_alignment(R_p, t_p)
    cmd = generate_forklift_command(fa)
    print(f'\n  [{label}]')
    print(f'    Errors: {fa.message}')
    print(f'    Cmd → fwd={cmd.forward:+.3f}  lat={cmd.lateral:+.3f}  '
          f'turn={cmd.turn:+.3f}  lift={cmd.lift:+.3f}')
    print(f'    {cmd.message}')
print('═' * 75)

In [ ]:
# ── Simulate full approach sequence ────────────────────────────────────────

def simulate_approach(n_frames=50):
    history = []
    for i in range(n_frames):
        alpha = i / (n_frames - 1)
        z   = 1.5 - alpha * 1.0     # 1.5m → 0.5m
        x   = 0.12 * (1 - alpha)    # 12cm → 0cm lateral
        y   = 0.06 * (1 - alpha)    # 6cm → 0cm height error
        hdg = 0.15 * (1 - alpha)    # heading converges

        R_p = euler_to_R(0, hdg, 0)
        t_p = np.array([x, y, z])
        fa = compute_fork_alignment(R_p, t_p)
        cmd = generate_forklift_command(fa)
        history.append({'frame': i, 'fa': fa, 'cmd': cmd})

    return history

history = simulate_approach(50)

frames = [h['frame'] for h in history]
z_vals = [h['fa'].distance_m for h in history]
lat_vals = [h['fa'].lateral_error_m * 100 for h in history]
ht_vals = [h['fa'].height_error_m * 100 for h in history]
lift_vals = [h['cmd'].lift for h in history]
aligned = [h['fa'].is_aligned for h in history]

fig, axes = plt.subplots(2, 2, figsize=(14, 8))

# Distance
ax = axes[0, 0]
ax.plot(frames, z_vals, 'b-', lw=2)
ax.axhline(FORK_DOCK_DIST_M, color='green', ls='--', label=f'Target ({FORK_DOCK_DIST_M}m)')
ax.set_title('Distance to Pallet', fontweight='bold')
ax.set_ylabel('m'); ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

# Lateral error
ax = axes[0, 1]
ax.plot(frames, lat_vals, 'r-', lw=2)
ax.axhline(0, color='black', ls='--')
ax.axhspan(-FORK_TOL_LAT_M*100, FORK_TOL_LAT_M*100, alpha=0.15, color='green')
ax.set_title('Lateral Error', fontweight='bold')
ax.set_ylabel('cm'); ax.grid(True, alpha=0.3)

# Height error
ax = axes[1, 0]
ax.plot(frames, ht_vals, color='orange', lw=2)
ax.axhline(0, color='black', ls='--')
ax.axhspan(-FORK_TOL_HEIGHT_M*100, FORK_TOL_HEIGHT_M*100, alpha=0.15, color='green')
ax.set_title('Height Error (forks)', fontweight='bold')
ax.set_ylabel('cm'); ax.set_xlabel('Frame'); ax.grid(True, alpha=0.3)

# Lift command
ax = axes[1, 1]
ax.plot(frames, lift_vals, color='purple', lw=2)
ax.axhline(0, color='black', ls='--')
ax.set_title('Fork Lift Command', fontweight='bold')
ax.set_ylabel('m/s'); ax.set_xlabel('Frame'); ax.grid(True, alpha=0.3)
# Shade ALIGNED frames
if any(aligned):
    first_aligned = next(i for i, a in enumerate(aligned) if a)
    ax.axvspan(first_aligned, len(frames)-1, alpha=0.2, color='green', label='ALIGNED')
    ax.legend(fontsize=8)

plt.suptitle('Forklift Pallet Alignment: Full Approach Sequence', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

if any(aligned):
    first = next(i for i, a in enumerate(aligned) if a)
    print(f'\nFORKS ALIGNED at frame {first} (out of {len(frames)})')
else:
    print('\nNot yet aligned at end of sequence')

## 4 — Camera View Visualization

Project the fork pocket positions onto the camera image to show what the perception system "sees".

In [ ]:
def draw_pallet_overlay(img: np.ndarray, R_pallet: np.ndarray,
                        t_pallet: np.ndarray, K: np.ndarray,
                        fa: ForkAlignment) -> np.ndarray:
    """Draw pallet corners, pocket targets, and alignment info on image."""
    out = img.copy()
    dist = np.zeros(4)
    rvec, _ = cv2.Rodrigues(R_pallet)

    # Project pocket centers
    def proj(pt3d):
        p, _ = cv2.projectPoints(pt3d.reshape(1, 1, 3).astype(np.float32),
                                  rvec, t_pallet.astype(np.float32), K, dist)
        return tuple(p.reshape(2).astype(int))

    # Draw pocket targets
    left_2d  = proj(PALLET.left_pocket_center)
    right_2d = proj(PALLET.right_pocket_center)
    center_2d = proj(np.array([0, 0, 0]))

    cv2.circle(out, left_2d,  15, (0, 255, 100), 2)
    cv2.circle(out, right_2d, 15, (0, 255, 100), 2)
    cv2.line(out, left_2d, right_2d, (0, 200, 80), 1)
    cv2.putText(out, 'L', (left_2d[0]-7, left_2d[1]+5),
                cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 100), 2)
    cv2.putText(out, 'R', (right_2d[0]-7, right_2d[1]+5),
                cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 100), 2)

    # Draw coordinate axes at pallet center
    try:
        cv2.drawFrameAxes(out, K, dist, rvec, t_pallet.astype(np.float32), 0.10)
    except Exception:
        pass

    # Alignment info box
    color = (0, 200, 50) if fa.is_aligned else (0, 120, 255)
    cv2.rectangle(out, (5, 55), (320, 130), (20, 20, 30), -1)
    cv2.putText(out, f'Lat: {fa.lateral_error_m*100:+.1f}cm',
                (10, 75), cv2.FONT_HERSHEY_SIMPLEX, 0.55, color, 1)
    cv2.putText(out, f'Ht:  {fa.height_error_m*100:+.1f}cm',
                (10, 97), cv2.FONT_HERSHEY_SIMPLEX, 0.55, color, 1)
    cv2.putText(out, f'Dist: {fa.distance_m:.3f}m  Hdg: {fa.heading_error_deg:+.1f}deg',
                (10, 119), cv2.FONT_HERSHEY_SIMPLEX, 0.45, color, 1)

    if fa.is_aligned:
        h, w = out.shape[:2]
        cv2.putText(out, 'FORKS ALIGNED', (w//2 - 130, h//2),
                    cv2.FONT_HERSHEY_SIMPLEX, 1.2, (0, 255, 80), 3)

    return out

# Render 3 approach stages
stages = [
    (euler_to_R(0, 0.10, 0), np.array([0.10, 0.05, 1.40]), 'Far Approach'),
    (euler_to_R(0, 0.04, 0), np.array([0.03, 0.02, 0.80]), 'Close Alignment'),
    (euler_to_R(0, 0.01, 0), np.array([0.005, 0.008, 0.50]), 'ALIGNED'),
]

K = np.array([[600, 0, 320], [0, 600, 240], [0, 0, 1]], dtype=np.float64)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for ax, (R_p, t_p, label) in zip(axes, stages):
    frame = np.full((480, 640, 3), (30, 30, 40), dtype=np.uint8)
    # Draw pallet face
    cv2.rectangle(frame, (180, 160), (460, 320), (110, 80, 40), -1)
    cv2.rectangle(frame, (180, 160), (460, 320), (160, 120, 60), 2)
    # Draw pocket rectangles
    cv2.rectangle(frame, (210, 230), (270, 300), (15, 15, 25), -1)
    cv2.rectangle(frame, (370, 230), (430, 300), (15, 15, 25), -1)

    fa = compute_fork_alignment(R_p, t_p)
    overlay = draw_pallet_overlay(frame, R_p, t_p, K, fa)

    # Header
    cv2.rectangle(overlay, (0, 0), (640, 50), (20, 20, 30), -1)
    cv2.putText(overlay, label, (10, 33),
                cv2.FONT_HERSHEY_SIMPLEX, 0.8,
                (0, 200, 50) if fa.is_aligned else (0, 120, 255), 2)

    ax.imshow(cv2.cvtColor(overlay, cv2.COLOR_BGR2RGB))
    ax.set_title(label, fontweight='bold')
    ax.axis('off')

plt.suptitle('Pallet Detection: Camera View with Fork Pocket Targets', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## Recap

**Next:** `25_capstone_template.ipynb` — build your own robotics pose estimation project from scratch using the patterns from NB 23 and NB 24 as your reference.

| Concept | Key Point |
|---------|----------|
| Pallet geometry | Euro pallet: 1200×800×144mm, pocket sep = 550mm |
| 3 ArUco markers | Redundancy + averaging for robust pose estimation |
| Multi-marker averaging | Transform each marker pose to pallet center, average translations + re-orthogonalize R |
| Fork pocket 3D positions | `transform_point(pocket_pallet_frame, R_pallet, t_pallet)` |
| Fork alignment errors | 4 axes: lateral, height, distance, heading |
| Forklift command | P-controller adds **lift** axis compared to simple robot docking |
| Approach speed | Reduce max speed near pallet (safety: 0.15 m/s max close range) |
| Height tolerance | Tighter than lateral (1.5cm) — forks must match pocket height precisely |

---

> **Run this on a real forklift camera:**
> The complete script is at `scripts/forklift_pallet_alignment.py`.
> ```
> python scripts/forklift_pallet_alignment.py --calib assets/calibration/camera_calibration.npz
> ```
> Edit `PALLET_MARKER_IDS` and `MARKER_POS_IN_PALLET` at the top of the script for custom pallet layouts.

In [ ]:
# ============================================================
# EXERCISE 1: Adapt for a non-standard pallet
# ============================================================
# Create a PalletModel for an industrial rack pallet:
#   - Size: 1100 x 1100 x 150 mm
#   - Fork pocket separation: 700 mm
#   - Pocket opening: 200 x 120 mm
# What is the left pocket center in pallet frame?

# YOUR CODE HERE
# rack_pallet = PalletModel(...)
# print(rack_pallet.left_pocket_center * 1000, 'mm')

# ============================================================
# SOLUTION (uncomment to reveal)
# ============================================================
# rack_pallet = PalletModel(
#     width_m=1.100, depth_m=1.100, height_m=0.150,
#     pocket_sep_m=0.700, pocket_width_m=0.200, pocket_height_m=0.120,
#     pocket_depth_m=1.100
# )
# print('Left pocket:', rack_pallet.left_pocket_center * 1000, 'mm')
# # Expected: [-350, -60, 0] mm

In [ ]:
# ============================================================
# EXERCISE 2: Add "slow final advance" logic
# ============================================================
# After forks are aligned (is_aligned=True), the forklift still
# needs to advance slowly until forks are fully inserted
# (distance < 0.05m = forks fully in).
# Modify generate_forklift_command() to output:
#   forward = 0.05 m/s when is_aligned=True AND distance > 0.05m
#   forward = 0.0 when is_aligned=True AND distance <= 0.05m

# YOUR CODE HERE

# ============================================================
# SOLUTION (uncomment to reveal)
# ============================================================
# def generate_forklift_command_v2(fa: ForkAlignment) -> ForkliftCommand:
#     if fa.is_aligned:
#         if fa.distance_m > 0.05:  # advance until forks fully in
#             return ForkliftCommand(0.05, 0.0, 0.0, 0.0, 'ADVANCING — INSERTING FORKS')
#         else:
#             return ForkliftCommand(0.0, 0.0, 0.0, 0.0, 'FORKS FULLY INSERTED — STOP')
#     return generate_forklift_command(fa)  # normal alignment
#
# # Test
# fa_near = ForkAlignment(0.001, 0.001, 0.10, 0.5, np.zeros(3), np.zeros(3), True)
# cmd = generate_forklift_command_v2(fa_near)
# print(f'forward={cmd.forward:.2f}  msg={cmd.message}')